# 🚀 OmniAnomaly coreX v2 — Full GPU Training

## Instructions:
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run each cell in order.
3. Cell 1 will **restart** the session — this is normal.
4. After restart, **skip Cell 1** and run from Cell 2.

> ⚠️ Do NOT close this tab during training.

## Cell 0: Upload Project Zip

In [ ]:
from google.colab import files
import zipfile, os, shutil

print('--- Upload your coreX_project.zip ---')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')

os.chdir('/content/project')

# Fix Windows backslashes in filenames
for f in os.listdir('.'):
    if '\\' in f:
        parts = f.split('\\')
        dir_name = os.path.join(*parts[:-1])
        if dir_name:
            os.makedirs(dir_name, exist_ok=True)
        shutil.move(f, os.path.join(*parts))

print(f'Working in: {os.getcwd()}')
print('Files:', sorted([x for x in os.listdir('.') if not x.startswith('.')]))

## Cell 1: Install condacolab
⚠️ This **restarts the session**. After restart, skip this cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## Cell 2: Full Environment Setup
Creates Python 3.6 env, installs CUDA 9.0, TF 1.12, and ALL dependencies in one go.

In [ ]:
import os
os.chdir('/content/project')
print(f'Working in: {os.getcwd()}')

# 1. Create conda env
!conda create --name corex_env python=3.6 -y -q

# 2. Install CUDA from anaconda channel
!conda run -n corex_env conda install cudatoolkit=9.0 cudnn=7.1.2 -c anaconda -y -q

# 3. Install all pip dependencies at once
!conda run -n corex_env pip install --default-timeout=1000 --no-build-isolation \
    tensorflow-gpu==1.12.0 \
    tensorflow_probability==0.5.0 \
    scikit-learn==0.20.2 pandas==0.23.4 numpy==1.15.4 scipy==1.2.0 \
    matplotlib==3.0.2 tqdm==4.28.1 six==1.11.0 \
    imageio==2.4.1 fs==2.3.0 click==7.0 seaborn \
    git+https://github.com/haowen-xu/tfsnippet.git@v0.2.0-alpha1 \
    git+https://github.com/thu-ml/zhusuan.git

print('\n✅ Environment fully ready!')

## Cell 3: GPU Check

In [ ]:
!conda run -n corex_env python -c "import tensorflow as tf; print('GPU:', tf.test.is_gpu_available()); print('TF:', tf.__version__)"

## Cell 4: Apply All Code Patches + Preprocess + Train (50 Epochs)
This single cell patches all TF1.12 incompatibilities, runs preprocessing, and starts training.

Uses `bash -c 'source activate'` instead of `conda run` for real-time output.

In [ ]:
import os
os.chdir('/content/project')

# === PATCH 1: Fix tf.compat.v1 in main.py ===
with open('main.py', 'r') as f:
    content = f.read()
content = content.replace(
    'tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)',
    'tf.logging.set_verbosity(tf.logging.ERROR)'
)
content = content.replace('max_epoch = 100', 'max_epoch = 50')
content = content.replace("'max_epoch': 100,", "'max_epoch': 50,")
with open('main.py', 'w') as f:
    f.write(content)
print('✅ main.py patched (TF1.12 logging + 50 epochs)')

# === PATCH 2: Fix tf.math.reduce_std in model.py ===
with open('omni_anomaly/model.py', 'r') as f:
    content = f.read()

old = """                    z_mean = tf.reduce_mean(z_samples, axis=0)
                    z_std = tf.math.reduce_std(z_samples, axis=0)"""

new = """                    z_mean = tf.reduce_mean(z_samples, axis=0)
                    z_mean_temp = tf.reduce_mean(z_samples, axis=0, keepdims=True)
                    z_std = tf.sqrt(tf.reduce_mean(tf.square(z_samples - z_mean_temp), axis=0))"""

content = content.replace(old, new)
with open('omni_anomaly/model.py', 'w') as f:
    f.write(content)
print('✅ model.py patched (TF1.12 reduce_std fix)')

# Verify
with open('omni_anomaly/model.py', 'r') as f:
    txt = f.read()
    assert 'z_mean_temp' in txt, '❌ model.py patch FAILED'
    assert 'tf.math.reduce_std' not in txt, '❌ old code still present'
print('✅ All patches verified!')

## Cell 5: Run Preprocessing

In [ ]:
%%bash
cd /content/project
source activate corex_env
export PYTHONPATH=.
export MPLBACKEND=Agg
python -u data_preprocess.py

## Cell 6: 🚀 Train for 50 Epochs on GPU
Output streams in real-time. Training takes ~1-2.5 hours.

In [ ]:
%%bash
cd /content/project
source activate corex_env
export PYTHONPATH=.
export MPLBACKEND=Agg
python -u main.py

## Cell 7: Download Results
⚠️ Run this BEFORE closing the tab!

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('coreX_model', 'zip', '.', 'model_coreX_v1')
files.download('coreX_model.zip')

shutil.make_archive('coreX_results', 'zip', '.', 'results')
files.download('coreX_results.zip')

print('✅ Download complete!')